# BirdCLEF 2024 — Day 4: Train Long-Context CNN

**Kaggle notebook**: Trains the Long-Context CNN (LCC) on 10-second windows.

**Architecture (from Chapter 4.2 of the thesis)**:
- Backbone: `efficientnet_b0` (ImageNet pretrained).
- Input: $128 \times 640$ log-mel spectrogram of a 10-second audio window.
- Replaces ImageNet classifier with global mean pool + single linear layer (182 outputs).

**Training Protocol**:
- Loss: Multi-label BCE
- Sampler: Class-balanced sampling
- Augmentation: SpecAugment (2 time masks width 30, 2 freq masks width 16)
- Optimizer: AdamW ($lr=3 \times 10^{-4}$, weight decay $10^{-4}$)
- Schedule: Cosine over 30 epochs
- Batch size: 32

## 0. Environment

In [1]:
import os
import torch
import torch.nn as nn
import torchaudio
import timm
import pandas as pd
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG_MODE: set True when running locally.
# On Kaggle: set False → runs actual model training.
# ─────────────────────────────────────────────────────────────────────────────

import librosa

INPUT_DIR  = Path("/kaggle/input")
OUTPUT_DIR = Path(".")

EPOCHS =30
BATCH_SIZE =32
LR = 3e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

C:\Users\piyus\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. LCC Model Architecture

In [2]:
class LongContextCNN(nn.Module):
    def __init__(self, num_classes=182, pretrained=True):
        super().__init__()
        # Use timm to build EfficientNet-B0 with 1 input channel
        self.backbone = timm.create_model(
            "efficientnet_b0", 
            pretrained=pretrained, 
            num_classes=num_classes, 
            in_chans=1
        )
        # timm efficientnet automatically replaces the head with a pooling + linear layer
        # if num_classes is specified.
        
    def forward(self, x):
        return self.backbone(x)

## 2. Data & Augmentation Pipeline

In [3]:
class SpecAugment(nn.Module):
    def __init__(self, freq_mask_param=16, time_mask_param=30):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param)
        
    def forward(self, x):
        # x is (B, C, F, T)
        # Apply 2 frequency masks and 2 time masks
        for _ in range(2):
            x = self.freq_mask(x)
            x = self.time_mask(x)
        return x

class LCCDataset(Dataset):
    def __init__(self, df, audio_dir, is_train=True):
        self.df = df
        self.audio_dir = audio_dir
        self.is_train = is_train
        
        # Log-mel params: 10s @ 16kHz
        # Freq bins = 128, Time frames ~ 640 (hop 250)
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000, 
            n_fft=1024, 
            hop_length=250, 
            n_mels=128, 
            f_min=50.0, 
            f_max=8000.0
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
        self.spec_aug = SpecAugment() if is_train else nn.Identity()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = self.audio_dir / row["filename"]
        
        # Load audio, pad/crop to 10 seconds (16000 * 10)
        y, _ = librosa.load(audio_path, sr=16000, mono=True)
        
    target_len = 16000 * 10
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        # Random crop for training, center crop for valid
        if self.is_train:
            start = np.random.randint(0, len(y) - target_len + 1)
        else:
            start = (len(y) - target_len) // 2
        y = y[start : start + target_len]
        
    y_t = torch.tensor(y)
    
    # Transform: (1, 128, 640)
    mel = self.mel_transform(y_t)
    mel_db = self.amplitude_to_db(mel)
    mel_db = mel_db.unsqueeze(0) # add channel dim
    
    if self.is_train:
        mel_db = self.spec_aug(mel_db)
        
    return mel_db, torch.tensor(row["label_vec"], dtype=torch.float32)

## 3. Training Loop

In [4]:
if __name__ == "__main__":
    meta_df = pd.read_csv(INPUT_DIR / "birdclef-2024/train_metadata.csv")
    ALL_SPECIES = sorted(meta_df["primary_label"].unique())
    N_CLASSES = len(ALL_SPECIES)
    sp_to_idx = {sp: i for i, sp in enumerate(ALL_SPECIES)}
    
    # Build multi-hot vectors
    labels = []
    for _, row in meta_df.iterrows():
        vec = np.zeros(N_CLASSES)
        vec[sp_to_idx[row["primary_label"]]] = 1.0
        # Secondary labels (half weight)
        for sec in eval(row.get("secondary_labels", "[]")):
            if sec in sp_to_idx:
                vec[sp_to_idx[sec]] = 0.5
        labels.append(vec)
    meta_df["label_vec"] = labels
    train_df = meta_df

# Class-balanced sampler weights
counts = train_df["primary_label"].value_counts()
sample_weights = train_df["primary_label"].map(lambda sp: 1.0 / counts[sp]).values
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# Dataloader
AUDIO_DIR = INPUT_DIR / "birdclef-2024/train_audio"
train_dataset = LCCDataset(train_df, AUDIO_DIR, is_train=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)

# Model, Loss, Optimizer
model = LongContextCNN(num_classes=N_CLASSES, pretrained=not DEBUG_MODE).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("Starting LCC Training...")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
        
    scheduler.step()
    print(f"Epoch {epoch+1} Avg Loss: {epoch_loss/len(train_loader):.4f}")

# Save
torch.save(model.state_dict(), OUTPUT_DIR / "lcc_10s_wg.pth")
print("Training complete. Saved lcc_10s_wg.pth")

DEBUG_MODE=True — Generating mock training setup.
Starting LCC Training...


Epoch 1/2:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 1/2:   0%|          | 0/25 [00:01<?, ?it/s, loss=0.721]

Epoch 1/2:   4%|▍         | 1/25 [00:01<00:37,  1.58s/it, loss=0.721]

Epoch 1/2:   4%|▍         | 1/25 [00:02<00:37,  1.58s/it, loss=0.678]

...

Epoch 2/2:  96%|█████████▌| 24/25 [00:22<00:00,  1.05it/s, loss=0.108]

Epoch 2/2:  96%|█████████▌| 24/25 [00:23<00:00,  1.05it/s, loss=0.105]

Epoch 2/2: 100%|██████████| 25/25 [00:23<00:00,  1.04it/s, loss=0.105]

Epoch 2/2: 100%|██████████| 25/25 [00:23<00:00,  1.06it/s, loss=0.105]

Epoch 2 Avg Loss: 0.1321
Training complete. Saved lcc_10s_wg.pth
